In [0]:
from pyspark.sql import SparkSession

# Create SparkSession
spark = SparkSession.builder \
    .appName("MySparkApp") \
    .getOrCreate()

print(f"Spark version: {spark.version}")

In [0]:
%sql
select * from samples.bakehouse.media_gold_reviews_chunked

In [0]:
df = spark.read.csv("/databricks-datasets/Rdatasets/data-001/csv/ggplot2/diamonds.csv", header=True)
# df.show()
display(df)
df.write.mode("overwrite").saveAsTable("mytable")

In [0]:
display(dbutils.fs.ls('/'))
display(dbutils.fs.ls('/databricks-datasets/'))

In [0]:
# can directly read tables as well
tableDF = spark.read.table('samples.accuweather.forecast_daily_calendar_imperial')
display(tableDF)

In [0]:
from pyspark.sql.types import *

schema = StructType([
    StructField('user_id', IntegerType(), True),
    StructField('name', StringType(), True),
    StructField('age', IntegerType(), True),
    StructField('salary', DoubleType(), True)
])

df = (
    spark.read
    .option("header", True)
    .schema(schema)
    .csv('/Volumes/workspace/default/mydata/salary.csv')
)

display(df)

In [0]:
# read 1st file consiting of header row
headerDF = (
    spark.read
    .option("header", "true")
    # .option("inferSchema", "true")
    .csv("/Volumes/samples/databricks/datasets/airlines/part-00000")
)
# display(headerDF)

# extract column names
columns = headerDF.columns

# fetch the list of files having no header row, exclude the first file
files = dbutils.fs.ls('/Volumes/samples/databricks/datasets/airlines/')
remaining = [f.path for f in files if f.name != 'part-00000']

# read all remaining files having no header row
dataDF = (
    spark.read
    .csv(remaining)
    .toDF(*columns)
)
# display(dataDF)

# union both dfs for complete data
finalDF = headerDF.unionByName(dataDF)
display(finalDF)




In [0]:
orders.select(
    col("customer_id"),
    col("product_id"),
    (col("qty") * col("price")).alias("total_amount")
)

In [0]:
# filtering - filter()
df.filter(condition)
df.where(condition)

# .filter() and .where() are same technically, just alias to each other

df.filter(c1 & c2)
# & -- and 
# | -- or
# ~ -- not

df.filter(c1.isNull())
df.filter(c1.isNotNull())

df.filter(c1.isin(list of options))

In [0]:
from pyspark.sql.functions import col, when

data = [
    (1,'Alice',50000, 'IT', True),
    (2,'Bob',80000, 'HR', True),
    (3,'Alice',50000, 'IT', True),
    (4,'David',120000, 'IT', False)
]

columns = ['id', 'name', 'salary', 'dept', 'is_active']

df = spark.createDataFrame(data, schema=columns)
display(df)

# 1. Create annual_salary from salary.
df = df.withColumn('annual_salary', col('salary') * 12)

# 2. Create salary_category:
# < 60000 → "Low"
# 60000–99999 → "Medium"
# >= 100000 → "High"
df = df.withColumn('salary_category', 
                   when(col('salary') < 60000, 'Low')
                   .when(col('salary') >= 100000, 'High')
                   .otherwise('Medium')
                )

# 3. Remove is_active.
df = df.drop('is_active')

# 4. Rename dept to department using alias().

# 5. Remove completely duplicate rows.
df = df.distinct()

# 6. Select only: id, name, department, annual_salary, salary_category
df = df.select('id', "name", col("dept").alias("department"), "annual_salary", "salary_category")

display(df)

In [0]:
from pyspark.sql.functions import lit, col, when

data = [
    (1, "Alice", "50000", "IT"),
    (2, "Bob", "80000", "HR"),
    (3, "Charlie", "120000", "IT"),
    (4, "David", "unknown", "Finance")
]

columns = ["id", "name", "salary", "department"]

df = spark.createDataFrame(data, columns)

display(df)

df = (
    df
    .withColumn('salary', col('salary').try_cast('integer'))
    .withColumn('annual_salary', col('salary') * 12)
    .withColumn('salary_category', 
                when(col('salary') < 60000, 'Low')
                .when(col('salary') >= 100000, 'High')
                .otherwise('Medium'))
    .withColumn('country', lit('India'))
    .withColumn('is_tech', 
                when(col('department').isin('IT', 'Engineering'), True)
                .otherwise(False))
    .filter(
        (col('salary') >= 80000) & (col('department').isin('IT', 'Engineering'))
    )
)

display(df)